# 00 · Prerequisites & Connectivity Check

Run this notebook **before** the workshop. Every section ends in a ✅ / ❌ result and a plain-English fix.
It checks, in order:

1. Python runtime and `neocarta[snowflake]` install (`uv` preferred, `pip` fallback)
2. Configuration file (`.env`) and URI/protocol sanity
3. Neo4j connectivity (Aura vs. Desktop/self-hosted)
4. Snowflake connectivity (PAT vs. password)
5. Access to the target dataset (`SNOWFLAKE_SAMPLE_DATA.TPCH_SF1` or your own schema)
6. `neocarta` ingestion readiness

> Nothing here writes to Neo4j or Snowflake. Section 6 only constructs the connector.

## 1 · Environment setup

Preferred (`uv`), from the repository root:
```sh
uv sync --extra dev
source .venv/bin/activate        # optional; Windows: .venv\Scripts\activate
```
Fallback (plain `pip`):
```sh
python -m venv .venv && source .venv/bin/activate
pip install -e ".[dev]"
```
Then launch this notebook **from that environment** (`uv run jupyter lab`).

In [ ]:
import importlib.util, shutil, sys
from importlib import metadata

results = []   # (check, ok, detail)
def record(check, ok, detail=""):
    results.append((check, ok, detail))
    print(("✅" if ok else "❌"), check, ("— " + detail) if detail else "")

record("Python >= 3.11", sys.version_info >= (3, 11), sys.version.split()[0])
record("Running inside a virtualenv", sys.prefix != sys.base_prefix,
       sys.prefix if sys.prefix != sys.base_prefix else "launch Jupyter from your .venv")
print("ℹ️  uv:", shutil.which("uv") or "not found (pip fallback is fine)")

for pkg in ["neocarta", "neo4j", "snowflake-connector-python", "pandas", "python-dotenv"]:
    try:
        record(f"package {pkg}", True, metadata.version(pkg))
    except metadata.PackageNotFoundError:
        record(f"package {pkg}", False, 'install with: uv sync --extra dev')

## 2 · Configuration

Copy the template and fill it in:
```sh
cp .env.example .env
```
`.env` is gitignored. See `docs/attendee_guide.md` for where to find each value.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None
    record("python-dotenv importable", False, "uv pip install python-dotenv")

env_file = next((p for p in (Path(".env"),) if p.exists()), None)
if env_file and load_dotenv:
    load_dotenv(env_file, override=True)
record("config file found", env_file is not None, str(env_file) if env_file else "copy .env.example to .env")

REQUIRED = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER",
            "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA"]
PLACEHOLDERS = ("xxxxxxxx", "your-", "xy12345", "workshop_user")
for k in REQUIRED:
    v = os.getenv(k, "")
    record(f"{k} set", bool(v) and not any(p in v for p in PLACEHOLDERS),
           "" if v and not any(p in v for p in PLACEHOLDERS) else "missing or still the template value")

# --- Neo4j URI / protocol sanity -------------------------------------------------
uri = os.getenv("NEO4J_URI", "")
scheme = urlparse(uri).scheme
host = urlparse(uri).hostname or ""
is_aura = host.endswith(".databases.neo4j.io")
if is_aura and scheme not in ("neo4j+s", "neo4j+ssc"):
    record("Neo4j URI scheme", False, f"Aura requires neo4j+s:// (or neo4j+ssc:// behind a TLS-inspecting proxy), got {scheme}://")
elif host in ("localhost", "127.0.0.1") and scheme in ("neo4j+s", "neo4j+ssc", "bolt+s"):
    record("Neo4j URI scheme", False, f"local servers normally have no TLS; use neo4j:// or bolt://, got {scheme}://")
elif scheme in ("neo4j", "neo4j+s", "neo4j+ssc", "bolt", "bolt+s", "bolt+ssc"):
    record("Neo4j URI scheme", True, f"{scheme}:// ({'Aura' if is_aura else 'self-hosted'})")
else:
    record("Neo4j URI scheme", False, f"unrecognised scheme {scheme!r}")

# --- Snowflake auth method -------------------------------------------------------
authenticator = os.getenv("SNOWFLAKE_AUTHENTICATOR", "").upper()
if os.getenv("SNOWFLAKE_PRIVATE_KEY_PATH"):
    auth_mode = "keypair"
elif authenticator:
    auth_mode = "authenticator"
elif os.getenv("SNOWFLAKE_PASSWORD"):
    auth_mode = "password"
else:
    auth_mode = None
record("Snowflake auth method configured", auth_mode is not None,
       f"{auth_mode} (precedence: key-pair > authenticator > password)" if auth_mode else "set SNOWFLAKE_TOKEN+AUTHENTICATOR, or SNOWFLAKE_PASSWORD")
if auth_mode == "authenticator" and authenticator == "PROGRAMMATIC_ACCESS_TOKEN" and not os.getenv("SNOWFLAKE_TOKEN"):
    record("SNOWFLAKE_TOKEN set for PAT", False, "authenticator is PROGRAMMATIC_ACCESS_TOKEN but SNOWFLAKE_TOKEN is empty")
if os.getenv("SNOWFLAKE_AUTHENTICATOR") and os.getenv("SNOWFLAKE_PASSWORD"):
    print("⚠️  Both authenticator and password are set; the authenticator wins. Remove the one you don't want.")

## 3 · Neo4j connectivity

| Target | URI | Notes |
|---|---|---|
| Aura Free / Pro / VDC | `neo4j+s://<id>.databases.neo4j.io` | Use `neo4j+ssc://` only if a corporate proxy re-signs TLS |
| Desktop / local / self-hosted | `neo4j://localhost:7687` or `bolt://localhost:7687` | Make sure the DBMS is **started** |

Aura instances created from the console may be paused or still provisioning — check the console if this times out.

In [ ]:
from neo4j import GraphDatabase
from neo4j.exceptions import AuthError, ServiceUnavailable

neo4j_ok = False
try:
    with GraphDatabase.driver(uri, auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))) as driver:
        driver.verify_connectivity()
        db = os.getenv("NEO4J_DATABASE", "neo4j")
        rec = driver.execute_query("CALL dbms.components() YIELD name, versions, edition RETURN name, versions[0] AS v, edition",
                                   database_=db).records[0]
        record("Neo4j connect + auth", True, f"{rec['name']} {rec['v']} ({rec['edition']}), database={db}")
        writable = driver.execute_query("RETURN 1 AS ok", database_=db).records[0]["ok"] == 1
        neo4j_ok = writable
except AuthError:
    record("Neo4j connect + auth", False, "bad username/password (Aura: the password is shown only once at creation)")
except ServiceUnavailable as e:
    record("Neo4j connect + auth", False, f"cannot reach {uri}: check URI scheme, instance running/not paused, VPN/proxy. {e}")
except Exception as e:
    record("Neo4j connect + auth", False, f"{type(e).__name__}: {e}")

## 4 · Snowflake connectivity

**Free-trial accounts and PATs.** Snowflake requires a *network policy* before a Programmatic Access Token can be used.
For a workshop trial account, when generating the token choose the option that **temporarily bypasses the network policy requirement**
(Snowsight → your user → *Programmatic access tokens* → *Generate new token* → *Bypass network policy temporarily* / "temporary access grant").
Set an expiry of a few days.
If you can't do that, fall back to **password auth** (`SNOWFLAKE_PASSWORD`, remove `SNOWFLAKE_AUTHENTICATOR`/`SNOWFLAKE_TOKEN`) —
this fails on MFA-enforced accounts, where key-pair auth (`SNOWFLAKE_PRIVATE_KEY_PATH`) is required.

In [ ]:
import snowflake.connector

def snowflake_connect():
    """Build a connection using the same precedence as neocarta: key-pair > authenticator > password."""
    kw = dict(account=os.getenv("SNOWFLAKE_ACCOUNT"), user=os.getenv("SNOWFLAKE_USER"),
              warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"), database=os.getenv("SNOWFLAKE_DATABASE"),
              schema=os.getenv("SNOWFLAKE_SCHEMA"), login_timeout=30, network_timeout=60)
    if os.getenv("SNOWFLAKE_ROLE"):
        kw["role"] = os.getenv("SNOWFLAKE_ROLE")
    if auth_mode == "keypair":
        from cryptography.hazmat.primitives import serialization
        pw = os.getenv("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")
        key = serialization.load_pem_private_key(Path(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"]).read_bytes(),
                                                 password=pw.encode() if pw else None)
        kw["private_key"] = key.private_bytes(serialization.Encoding.DER, serialization.PrivateFormat.PKCS8,
                                              serialization.NoEncryption())
    elif auth_mode == "authenticator":
        kw["authenticator"] = os.environ["SNOWFLAKE_AUTHENTICATOR"]
        if os.getenv("SNOWFLAKE_TOKEN"):
            kw["token"] = os.environ["SNOWFLAKE_TOKEN"]
    else:
        kw["password"] = os.getenv("SNOWFLAKE_PASSWORD")
    return snowflake.connector.connect(**kw)

sf = None
try:
    sf = snowflake_connect()
    cur = sf.cursor()
    cur.execute("SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()")
    acct, usr, role, wh, ver = cur.fetchone()
    record("Snowflake connect + auth", True, f"account={acct} user={usr} role={role} warehouse={wh} v{ver}")
    record("Warehouse selected", wh is not None, "" if wh else "set SNOWFLAKE_WAREHOUSE to a warehouse your role can USE")
except snowflake.connector.errors.DatabaseError as e:
    msg = str(e)
    hint = ("network policy blocks this token: regenerate the PAT with temporary network-policy bypass, or use password auth"
            if "network policy" in msg.lower() else
            "check SNOWFLAKE_ACCOUNT format (<org>-<account> or <locator>.<region>), credentials, and that the token has not expired")
    record("Snowflake connect + auth", False, f"{hint}. Raw error: {msg[:300]}")
except Exception as e:
    record("Snowflake connect + auth", False, f"{type(e).__name__}: {str(e)[:300]}")

## 5 · Target dataset access

Public workshop default is `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1` (8 tables). Trial accounts have it by default; if it's missing, in Snowsight run
`CREATE DATABASE SNOWFLAKE_SAMPLE_DATA FROM SHARE SFC_SAMPLES.SAMPLE_DATA;`.

Two separate permissions matter, and they affect the workshop differently:

- **Metadata** (`INFORMATION_SCHEMA`) — required for ingestion.
- **`SELECT` on the tables** — only needed for value sampling. If it is missing, set `SNOWFLAKE_VALUE_SAMPLE_LIMIT=0`.

Also note: **`REFERENCES` edges only appear if Snowflake has primary/foreign key (informational) constraints declared.**
`TPCH_SF1` declares none, so expect no `REFERENCES` edges there — that's expected, not a bug.

In [ ]:
from neocarta.connectors.snowflake._identifiers import normalize_identifier  # same case folding the connector uses

sample_limit = int(os.getenv("SNOWFLAKE_VALUE_SAMPLE_LIMIT", "10"))
db_id, schema_id = os.getenv("SNOWFLAKE_DATABASE"), os.getenv("SNOWFLAKE_SCHEMA")
tables = []
if sf:
    try:
        db_n, schema_n = normalize_identifier(db_id), normalize_identifier(schema_id)
        cur = sf.cursor()
        cur.execute(f"SELECT TABLE_NAME, TABLE_TYPE FROM {db_n}.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = %s ORDER BY 1", (schema_n.strip('"'),))
        tables = cur.fetchall()
        record("Metadata readable (INFORMATION_SCHEMA)", len(tables) > 0,
               f"{len(tables)} tables/views in {db_id}.{schema_id}: " + ", ".join(t[0] for t in tables[:8])
               if tables else f"no tables visible in {db_id}.{schema_id}: check names (stored case!) and role grants (USAGE on db+schema)")
    except Exception as e:
        record("Metadata readable (INFORMATION_SCHEMA)", False, f"{type(e).__name__}: {str(e)[:300]}")

    if tables:
        cur.execute(f"SHOW PRIMARY KEYS IN SCHEMA {db_n}.{schema_n}")
        pk = len(cur.fetchall())
        cur.execute(f"SHOW IMPORTED KEYS IN SCHEMA {db_n}.{schema_n}")
        fk = len(cur.fetchall())
        print(f"ℹ️  declared primary keys: {pk}, foreign keys: {fk}"
              + ("" if fk else "  → no REFERENCES edges will be created for this schema"))

        first = tables[0][0]
        try:
            cur.execute(f'SELECT * FROM {db_n}.{schema_n}."{first}" LIMIT 1')
            cur.fetchall()
            record("SELECT on table data (value sampling)", True, f"read {first}")
        except Exception as e:
            ok_zero = sample_limit == 0
            record("SELECT on table data (value sampling)", ok_zero,
                   "SELECT denied but SNOWFLAKE_VALUE_SAMPLE_LIMIT=0 so sampling is disabled — fine" if ok_zero
                   else f"SELECT denied on {first}: set SNOWFLAKE_VALUE_SAMPLE_LIMIT=0 or ask for SELECT. {str(e)[:200]}")
else:
    print("⏭️  skipped: Snowflake connection failed in section 4")

## 6 · Neocarta ingestion readiness

Constructs `SnowflakeSchemaConnector` against both live connections. **No ingestion is run.**

In [ ]:
if sf and neo4j_ok:
    try:
        from neocarta.connectors.snowflake import SnowflakeSchemaConnector
        driver = GraphDatabase.driver(uri, auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD")))
        connector = SnowflakeSchemaConnector(connection=sf, database=db_id, neo4j_driver=driver,
                                             database_name=os.getenv("NEO4J_DATABASE", "neo4j"),
                                             value_sample_limit=sample_limit)
        record("SnowflakeSchemaConnector constructs", True, f"value_sample_limit={sample_limit}")
        connector.close()
        driver.close()
    except Exception as e:
        record("SnowflakeSchemaConnector constructs", False, f"{type(e).__name__}: {e}")
else:
    record("SnowflakeSchemaConnector constructs", False, "skipped: fix Neo4j and/or Snowflake connectivity first")
if sf:
    sf.close()

## Summary

In [ ]:
width = max(len(c) for c, _, _ in results)
for c, ok, d in results:
    print(f"{'✅' if ok else '❌'} {c:<{width}}  {'' if ok else d}")
failed = [c for c, ok, _ in results if not ok]
print()
print("🎉 You're ready for the workshop." if not failed else f"⚠️  {len(failed)} check(s) need attention: fix them above and re-run from the top.")